In [51]:
#!pip install sympy

In [52]:
from pyomo.environ import *
import numpy as np
from math import pi
from Sympy_OPF_LALM_class import SympyACOPFModel
from Sympy_OPF_LALM_class import PrintQHDACOPFResults
from Sympy_OPF_LALM_class import solve_with_gurobi_from_sympy
from Sympy_OPF_LALM_class import extract_qhd_solution_vector
from Sympy_OPF_LALM_class import initialize_qhd_acopf_log
from Sympy_OPF_LALM_class import append_qhd_acopf_results
from qhdopt import QHD
import json

In [53]:
def load_matpower_json(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    Sbase = float(data["Sbase"])

    # Convert "k1", "k2", ... into integer keys 1, 2, ...
    buses = {int(k.replace("k", "")): v for k, v in data["buses"].items()}
    lines = {int(k.replace("k", "")): v for k, v in data["lines"].items()}
    gens  = {int(k.replace("k", "")): v for k, v in data["gens"].items()}

    return Sbase, buses, lines, gens


In [54]:
# Initialize the model.
# model = SympyACOPFModel()   # Enable the proximal option if needed later.
n = 2 # Select the test system size: 2, 3, or larger.

if n == 2:
    # 2-bus model
    Sbase = 10.0
    buses = {
        1: [1, 0, 1.00, 0.0, 0.0, 0.0, 0.0, 0.0],
        2: [2, 1, 1.01, 0.0, 0.0, 0.0, 0.3, 0.1],
    }
    lines = {
        1: [1, 2, 0.0452, 0.1852, 0.0204, 1.0, 30.0 / Sbase],
    }
    gens = {
        1: [1, 0.0 / Sbase, 20.0 / Sbase, -20.0 / Sbase, 100.0 / Sbase, 0.00375, 2.0, 0.0],
    }
    model = SympyACOPFModel(Sbase=Sbase, buses=buses, lines=lines, gens=gens)

elif n == 3:
    # 3-bus model
    model = SympyACOPFModel()

else:
    # n-bus model
    Sbase, buses, lines, gens = load_matpower_json(f"case{n}_custom_pretty.json")
    model = SympyACOPFModel(Sbase=Sbase, buses=buses, lines=lines, gens=gens)

print("Model initialized with", n, "buses", model.n_lines, "lines and", model.n_gens, "generators.")


Model initialized with 2 buses 1 lines and 1 generators.


In [55]:
# =========================
#   Linear ALM + QHD Loop
# =========================

import numpy as np

# 1) 构造 h(x)
h_func = model.build_h_func()
model.reset_lambdas(0.0)

# 2) 初始点
xk = model.build_initial_x0()
#xk = np.load("opf_result_ordered.npy", allow_pickle=True)

rho = 50
alpha = 0.3   # 对偶步长尽量小一点 simi 0.5-1.0 (2 bus)
max_outer = 1
tol = 1e-4
option = 1 # 1: QHD, 2: Gurobi
qhd_solver = "simbi"  # simbi / openjij / gurobi

# ========= 新增：初始化日志文件 =========
# alpha 调度选项
# alpha_option: "fixed" / "decay"
alpha_option = "decay"
fixed_alpha = alpha

# 若 alpha_option == "decay"，则有两种输入方式：
# 1. alpha0 + alpha_decay_factor -> alpha_(k+1) = alpha_k * alpha_decay_factor
# 2. alpha0 + alphaN -> alpha_decay_factor = (alphaN / alpha0)**(1 / max_outer)
# 若用户什么都没输入，则默认采用方案 2：alpha0=1, alphaN=1e-4
alpha0 = 20#None
alpha_decay_factor = None
alphaN = 1e-6 #None

if alpha_option == "fixed":
    if fixed_alpha <= 0:
        raise ValueError("fixed_alpha must be positive.")
    alpha = fixed_alpha
elif alpha_option == "decay":
    alpha0_use = 1.0 if alpha0 is None else alpha0
    if alpha0_use <= 0:
        raise ValueError("alpha0 must be positive.")
    if alpha_decay_factor is not None:
        if alpha_decay_factor <= 0:
            raise ValueError("alpha_decay_factor must be positive.")
        decay_factor = alpha_decay_factor
    else:
        alphaN_use = 1e-4 if alphaN is None else alphaN
        if alphaN_use <= 0:
            raise ValueError("alphaN must be positive.")
        decay_factor = (alphaN_use / alpha0_use) ** (1.0 / max_outer)
    alpha = alpha0_use
else:
    raise ValueError("alpha_option must be 'fixed' or 'decay'.")

log_file = initialize_qhd_acopf_log(model, folder="logs", option=option, qhd_solver=qhd_solver)
print("Log file:", log_file)
print(f"Alpha option: {alpha_option}")
if alpha_option == "fixed":
    print(f"Fixed alpha: {fixed_alpha}")
else:
    print(f"Decay alpha: alpha0={alpha0_use}, decay_factor={decay_factor}")

print("\n===== Start Linear ALM Loop =====\n")

for k in range(max_outer):

    if alpha_option == "fixed":
        alpha = fixed_alpha
    else:
        alpha = alpha0_use * (decay_factor ** k)

    print(f"\n--- Outer Iteration {k} ---")
    print(f"alpha = {alpha:.6e}")

    # ======================================
    # (1) 在线性化点 xk 构造二次 L^(k)(x)
    # ======================================
    Lag, variable_list, Var_bound_list = \
        model.build_linear_ALM_Lagrangian_syms(
            x_center=xk,
            rho=rho,
            ref_bus_id=None,
            mu_prox=0
        )
    
    bad_bounds = []
    for i, (var, bnd) in enumerate(zip(variable_list, Var_bound_list)):
        lb, ub = float(bnd[0]), float(bnd[1])
        if ub < lb:
            bad_bounds.append((i, str(var), lb, ub))

    if bad_bounds:
        print("\n=== Invalid bounds found ===")
        for item in bad_bounds:
            print(item)
        raise ValueError("Var_bound_list contains invalid bounds (ub < lb).")

    if option == 1:
        # ======================================
        # (2A) QHD 求解子问题
        # ======================================
        qhd_model = QHD.SymPy(Lag, variable_list, Var_bound_list)


        if qhd_solver == "simbi":
            qhd_model.simbi_setup(
                resolution=16,
                agents=1024,
                max_steps=5000,
                embedding_scheme="unary",
                post_processing_method='TNC',
                #ballistic=True,
                #heated=True,
                best_only=False,
                verbose=True
            )
        elif qhd_solver == "openjij":
            qhd_model.openjij_setup(
                resolution=6,
                shots=2048,
                sampler_name="SQASampler",
                post_processing_method='TNC',
                seed=42,
                debug=False,
                sampler_init_kwargs={},
                sample_kwargs={
                    "beta": 5.0,
                    "gamma": 1.0,
                    "trotter": 8,
                    "num_sweeps": 10000,
                    "reinitialize_state": True,
                },
            )
        elif qhd_solver == "gurobi":
            qhd_model.gurobi_setup(
                resolution=4,
                shots=20,
                embedding_scheme="unary",
                solver_mode="ising",
                time_limit=30,
                threads=0,
                log_to_console=False,
                post_processing_method='TNC',
            )
        else:
            raise ValueError(f"Unsupported qhd_solver={qhd_solver!r}. Use 'simbi', 'openjij', or 'gurobi'.")
        response = qhd_model.optimize(refine=True, verbose=0)
        #response = qhd_model.optimize(refine=False, verbose=0)

        x_new = extract_qhd_solution_vector(response, expected_len=len(variable_list))

    else:
        # ======================================
        # (2B) 用 Gurobi 求解子问题
        # ======================================
        x_new = solve_with_gurobi_from_sympy(
        L_sym=Lag,
        variable_list=variable_list,
        Var_bound_list=Var_bound_list,
        verbose=False   # 如果想看 Gurobi 日志就改成 True
    )
    # ======================================
    # (3) 计算真实约束
    # ======================================
    h_val = h_func(x_new)
    norm_h = np.linalg.norm(h_val)

    print("||h(x)|| =", norm_h)

    # ======================================
    # (4) 对偶更新
    # ======================================
    lambda_new, h_x = model.update_lambda(
        x_new,
        alpha=alpha,
        h_func=h_func
    )
    # ======================================
    # (5) 自适应 rho
    # ======================================
    h_old = h_func(xk)
    print(f"[rho-check] ||h_old||={np.linalg.norm(h_old):.3e}, ||h_new||={np.linalg.norm(h_val):.3e}, rho={rho:.3g}")
    # Keep rho fixed for stability diagnostics.
    print("Keeping rho fixed at", rho)

    # ======================================
    # (6) 可行性检查
    # ======================================
    _, check_flag = model.check_constraints(x_new)
    print("Constraint check:", check_flag)
    if check_flag:
        print("\nConverged!")
        break

    # ======================================
    # (7) 记录日志（每轮追加）
    # ======================================
    subs_dict = {var: val for var, val in zip(model.variable_list, x_new)}
    #objective_value = float(model.objective.evalf(subs=subs_dict))

    log_file = PrintQHDACOPFResults(
        model,
        x_new,
        log_file=log_file,
        iteration=k,
        folder="logs",
        print_to_console=True,
        rho=rho,
        alpha=alpha,
        h_x=h_val,
        lambda_vec=lambda_new,
        objective_value=0,
        feasibility=check_flag,
    )

    # 如果你还想同时在屏幕上打印表格，可以再保留这一句
    # PrintQHDACOPFResults(model, x_new, iteration=k, log_file=log_file, print_to_screen=True)


    # ======================================
    # (8) 收敛判据
    # ======================================
    step_norm = np.linalg.norm(x_new - xk)
    if norm_h < tol and step_norm < 1e-5:
        print("\nConverged!")
        break

    # ======================================
    # (9) 更新 primal
    # ======================================
    xk = x_new.copy()

print("\n===== End Loop =====\n")
print("Final log file:", log_file)

Log file: logs\Buses-2_04-24-2026_16-57-44.txt
Alpha option: decay
Decay alpha: alpha0=20, decay_factor=5e-08

===== Start Linear ALM Loop =====


--- Outer Iteration 0 ---
alpha = 2.000000e+01


Bifurcated agents:   0%|          | 0/1024 [00:02<?, ?it/s]


Minimizer: [0.     0.1875 0.9375 0.9375 0.5    0.5    0.625  0.8125 0.5    0.5
 0.5    0.5    0.     0.    ]
||h(x)|| = 0.11866446597828632
[rho-check] ||h_old||=4.543e-01, ||h_new||=1.187e-01, rho=50
Keeping rho fixed at 50
Constraint check: False
Update time: 2026-04-24 16:57:59
Iteration: 0
rho: 50
alpha: 20
objective_value: 0
feasible: False
max_abs_h: 7.856397714522e-02
l2_norm_h: 1.186644659783e-01
lambda_inf_norm: 1.571279542904e+00
lambda_l2_norm: 2.373289319566e+00
------------------------------------------------------------------------------------------------------------------------
Bus Results
BusID	V_R		V_I		Vmag		Pg		Qg		Pl		Ql
1	1.000001	0.000020	0.999460	0.142211	0.056025	0.000000	0.000000
2	0.975384	-0.037502	0.974837	0.000000	0.000000	0.300000	0.100000

Branch Results
LineID	From	To	Pik		Pki		Qik		Qki		LossP		LossQ
1	1	2	0.181510	-0.261616	0.056422	-0.100605	-0.080105	-0.044182


===== End Loop =====

Final log file: logs\Buses-2_04-24-2026_16-57-44.txt


In [56]:
len(list(Lag.free_symbols)),list(Lag.free_symbols)

(14,
 [Q_G0,
  Q_ij1,
  V_R0,
  V_sq0,
  P_G0,
  P_ij1,
  V_I1,
  V_I0,
  Q_ij0,
  S_tot_sq0,
  V_R1,
  V_sq1,
  P_ij0,
  S_tot_sq1])

In [57]:
qhd_model.SymPy.__sizeof__()

48

In [58]:
backend = qhd_model.qhd_base.backend

h = getattr(backend, "h", None)
J = getattr(backend, "J", None)


print("h:", h)
print("J:", J)
print("h shape:", len(h) if h is not None else None)   


h: [-1218.913448878318, -5.7899121093749955, -5.399228515624998, -5.008544921875, -4.617861328125002, -4.2271777343750045, -3.836494140625007, -3.445810546874995, -3.055126953124997, -2.6644433593749994, -2.2737597656250017, -1.8830761718749969, -1.4923925781250027, -1.1017089843750014, -0.7110253906249984, 1212.412511378318, -1393.201603175193, -166.40625, -152.34375, -138.28125, -124.21875, -110.15625, -96.09375, -82.03125, -67.96875, -53.90625, -39.84375, -25.78125, -11.71875, 2.34375, 16.40625, 1243.201603175193, -1307.1076065473978, -40.1813922112975, 14.011968949609944, 68.20533011051829, 122.39869127142666, 176.59205243233316, 230.785413593242, 284.9787747541494, 339.1721359150573, 393.36549707596475, 447.5588582368731, 501.75221939778055, 555.9455805586887, 610.1389417195966, 664.3323028805045, 1931.258517216605, -1262.105020258173, -42.3061940219816, -35.24022096098328, -28.17424789998485, -21.10827483898653, -14.0423017779881, -6.976328716989826, 0.08964434400861832, 7.155617